# HU02 · Descargar y explorar BRISC (análisis exploratorio)

**Proyecto:** Sistema Inteligente para clasificar y segmentar tumores cerebrales en MRI · **Sprint 1** · Fase CRISP-ML(Q): comprensión de los datos

Este notebook cubre los criterios de aceptación de HU02:
1. Dataset descargado y estructura verificada (`classification task`, `segmentation task` y `manifest.csv`).
2. Conteos por clase y por plano, comparados con el artículo (6.000 imágenes; 4.793 con máscara).
3. Gráficos del análisis exploratorio y ejemplos de imagen con su máscara.
4. Limitaciones registradas (sin ID de paciente; imágenes sin máscara).
5. Composición por clase documentada.
6. Confirmar si las imágenes **sin tumor** tienen máscara (el artículo dice 4.793 pares; Kaggle habla de 6.000 imágenes y máscaras).
7. Revisar el formato de las máscaras (¿binarias?) y los tamaños de imagen.

**Cómo ejecutarlo (en tu equipo):**
1. Descarga el dataset con `python descargar_brisc.py` (queda en la carpeta `datasets` de tu usuario, fuera de OneDrive). Si ya lo tienes en otra carpeta, escribe su ruta en `BRISC_ROOT_MANUAL` (primera celda de código).
2. Abre este notebook en VS Code (extensión Jupyter) o en Jupyter y usa **Run All**. También puedes correr los dos con `python ejecutar_sprint1.py`.
3. Los resultados quedan en `salida_sprint1/` (figuras, `eda_resumen.json`, partición y lote de prueba).

> Licencia del dataset según los metadatos de Kaggle: **CC BY 4.0** (hay que citar a los autores de BRISC en el informe).

In [ ]:
import os, re, json, glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

SEED = 42
MASK_THR = 127   # las máscaras traen valores intermedios: se consideran tumor los píxeles > 127
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ----------------------------- CONFIGURACION LOCAL -----------------------------
# Por defecto el dataset se busca en  ~/datasets  (lo deja alli descargar_brisc.py) y en  <proyecto>/data.
# Si lo tienes en otra carpeta, escribe aqui la ruta de la carpeta que contiene "classification task":
BRISC_ROOT_MANUAL = ""      # ejemplo: "D:/datasets/brisc2025"
# --------------------------------------------------------------------------------

# Carpeta de salida (figuras, resumen y particion): junto al notebook, en ./salida_sprint1
OUT = Path("salida_sprint1")
OUT.mkdir(parents=True, exist_ok=True)


def find_dirs(base, pattern, max_depth=5):
    """busca carpetas cuyo nombre coincida con el patron (sin distinguir mayusculas)"""
    base, out = Path(base), []
    if not base.exists():
        return out
    for dp, dn, fn in os.walk(base):
        depth = len(Path(dp).relative_to(base).parts)
        if depth > max_depth:
            dn[:] = []
            continue
        for d in dn:
            if re.fullmatch(pattern, d, flags=re.I):
                out.append(Path(dp) / d)
    return out


def candidatos():
    c = []
    if BRISC_ROOT_MANUAL:
        c.append(Path(BRISC_ROOT_MANUAL))
    if os.environ.get("BRISC_ROOT"):
        c.append(Path(os.environ["BRISC_ROOT"]))
    c.append(Path.home() / "datasets")          # aqui lo deja descargar_brisc.py (fuera de OneDrive)
    for p in [Path.cwd(), *Path.cwd().parents][:5]:
        c.append(p / "data")
    return c


CLS_DIRS = []
for base in candidatos():
    CLS_DIRS = find_dirs(base, r"classification[ _]task")
    if CLS_DIRS:
        break
assert CLS_DIRS, ("No encontre la carpeta 'classification task'. Ejecuta primero descargar_brisc.py "
                  "o escribe la ruta del dataset en BRISC_ROOT_MANUAL (arriba).")
CLS = CLS_DIRS[0]
ROOT = CLS.parent
SEG_DIRS = find_dirs(ROOT, r"segmentation[ _]task")
SEG = SEG_DIRS[0] if SEG_DIRS else None
print("Raiz del dataset:", ROOT)
print("Clasificacion:", CLS.name, "| Segmentacion:", SEG.name if SEG else "NO ENCONTRADA")

IMG_EXT = {".jpg", ".jpeg", ".png"}
CLASS_ALIASES = {"glioma": "glioma", "meningioma": "meningioma", "pituitary": "pituitary",
                 "no tumor": "no_tumor", "no_tumor": "no_tumor", "notumor": "no_tumor"}
CLASS_ES = {"glioma": "Glioma", "meningioma": "Meningioma", "pituitary": "Tumor de hipófisis", "no_tumor": "Sin tumor"}
CLASS_ORDER = ["glioma", "meningioma", "pituitary", "no_tumor"]
PLANE_ES = {"ax": "axial", "co": "coronal", "sa": "sagital"}


def parse_name(name):
    """brisc2025_<split>_<indice>_<codigo de tumor>_<plano>_<secuencia>.<ext> (segun el articulo)"""
    tok = re.split(r"[_ ]+", Path(name).stem)
    return (tok[3] if len(tok) > 3 else None, tok[4] if len(tok) > 4 else None)


def build_cls_table():
    rows = []
    for split_dir in sorted(CLS.iterdir()):
        if not split_dir.is_dir():
            continue
        split = split_dir.name.lower()
        for cdir in sorted(split_dir.iterdir()):
            if not cdir.is_dir():
                continue
            cls = CLASS_ALIASES.get(cdir.name.lower().replace("-", " "), cdir.name.lower())
            for p in sorted(cdir.iterdir()):
                if p.suffix.lower() in IMG_EXT:
                    code, plane = parse_name(p.name)
                    with Image.open(p) as im:
                        w, h = im.size
                    rows.append(dict(file=p.name, stem=p.stem, path=str(p.relative_to(ROOT)), split=split, cls=cls,
                                     code=code, plane=plane, width=w, height=h))
    return pd.DataFrame(rows)


def build_seg_table():
    rows = []
    if SEG is None:
        return pd.DataFrame(columns=["stem", "split", "image", "mask"])
    for split_dir in sorted(SEG.iterdir()):
        if not split_dir.is_dir():
            continue
        imgs = {p.stem: p for p in (split_dir / "images").iterdir() if p.suffix.lower() in IMG_EXT}
        msks = {p.stem: p for p in (split_dir / "masks").iterdir() if p.suffix.lower() in IMG_EXT}
        for s in sorted(set(imgs) | set(msks)):
            rows.append(dict(stem=s, split=split_dir.name.lower(),
                             image=str(imgs[s].relative_to(ROOT)) if s in imgs else None,
                             mask=str(msks[s].relative_to(ROOT)) if s in msks else None))
    return pd.DataFrame(rows)

# Cantidades que reporta el articulo de BRISC (Tabla 1 y Tabla 2)
EXPECTED_CLASS_SPLIT = {("glioma", "train"): 1147, ("glioma", "test"): 254,
                        ("meningioma", "train"): 1329, ("meningioma", "test"): 306,
                        ("pituitary", "train"): 1457, ("pituitary", "test"): 300,
                        ("no_tumor", "train"): 1067, ("no_tumor", "test"): 140}
EXPECTED_PLANE_SPLIT = {("train", "ax"): 1550, ("train", "co"): 1660, ("train", "sa"): 1790,
                        ("test", "ax"): 387, ("test", "co"): 316, ("test", "sa"): 297}
EXPECTED_SEG_PAIRS = 4793


## 1. Estructura de carpetas

In [ ]:
def resumen_carpetas(base, depth=3):
    base = Path(base)
    for dp, dn, fn in os.walk(base):
        d = len(Path(dp).relative_to(base).parts)
        if d > depth:
            dn[:] = []
            continue
        dn.sort()
        n = len([f for f in fn if Path(f).suffix.lower() in IMG_EXT])
        extra = [f for f in fn if Path(f).suffix.lower() not in IMG_EXT]
        print("  " * d + Path(dp).name + f"/   imagenes: {n}" + (f"   otros: {extra[:4]}" if extra else ""))


resumen_carpetas(ROOT)
manifests = list(ROOT.glob("manifest.*")) + list(ROOT.rglob("manifest.csv"))
print("\nmanifest encontrado:", sorted({str(m.relative_to(ROOT)) for m in manifests}))

## 2. Clasificación: conteos por clase, partición y plano
Se compara con lo que reporta el artículo. Cualquier diferencia debe explicarse.

In [ ]:
df_cls = build_cls_table()
print("Imágenes de clasificación:", len(df_cls), "| entrenamiento:", (df_cls.split == "train").sum(), "| prueba:", (df_cls.split == "test").sum())

obs = df_cls.groupby(["cls", "split"]).size()
cmp_cls = pd.DataFrame({"esperado (artículo)": pd.Series(EXPECTED_CLASS_SPLIT), "observado": obs}).fillna(0).astype(int)
cmp_cls["diferencia"] = cmp_cls["observado"] - cmp_cls["esperado (artículo)"]
display(cmp_cls)

obs_p = df_cls.groupby(["split", "plane"]).size()
cmp_pl = pd.DataFrame({"esperado (artículo)": pd.Series(EXPECTED_PLANE_SPLIT), "observado": obs_p}).fillna(0).astype(int)
cmp_pl["diferencia"] = cmp_pl["observado"] - cmp_pl["esperado (artículo)"]
display(cmp_pl)

tabla = pd.crosstab([df_cls.cls, df_cls.split], df_cls.plane, margins=True)
display(tabla)

### Código de tumor en el nombre del archivo → clase
El nombre trae un código (por ejemplo `gl`). Se infiere su significado a partir de las carpetas de clasificación y se comprueba que cada código corresponda a una sola clase.

In [ ]:
code_map = df_cls.groupby("code")["cls"].agg(lambda s: s.value_counts().idxmax())
pureza = df_cls.groupby("code")["cls"].nunique()
print(code_map.to_dict())
print("Códigos que mezclan clases:", pureza[pureza > 1].to_dict() or "ninguno")

## 3. Segmentación: pares imagen–máscara
Aquí se confirma **cuántas imágenes tienen máscara** y si las de la clase *sin tumor* están incluidas.

In [ ]:
df_seg = build_seg_table()
df_seg["code"] = df_seg.stem.map(lambda s: parse_name(s)[0])
df_seg["cls"] = df_seg.code.map(code_map)
df_seg["tiene_imagen"] = df_seg.image.notna()
df_seg["tiene_mascara"] = df_seg["mask"].notna()
pares = df_seg[df_seg.tiene_imagen & df_seg.tiene_mascara]
print("Archivos en la tarea de segmentación:", len(df_seg))
print("Pares imagen–máscara completos:", len(pares), "| artículo:", EXPECTED_SEG_PAIRS)
print("Imágenes sin máscara:", int((~df_seg.tiene_mascara).sum()), "| máscaras sin imagen:", int((~df_seg.tiene_imagen).sum()))
display(pd.crosstab(pares.cls, pares.split, margins=True))

# ¿las imágenes "sin tumor" tienen máscara?
sin_tumor_cls = set(df_cls[df_cls.cls == "no_tumor"].stem)
con_mascara = set(pares.stem)
print("\nImágenes 'sin tumor' (clasificación):", len(sin_tumor_cls))
print("De ellas con máscara en segmentación:", len(sin_tumor_cls & con_mascara))
print("Clases presentes en segmentación:", pares.cls.value_counts().to_dict())

### Calidad de las máscaras y tamaño del tumor
Porcentaje de la imagen que ocupa el tumor (dato clave para elegir la función de pérdida en el Sprint 2) y verificación de que las máscaras sean binarias.

In [ ]:
def stats_mascara(rel):
    m = np.asarray(Image.open(ROOT / rel).convert("L"))
    vals = tuple(int(v) for v in np.unique(m)[:6])
    return float((m > MASK_THR).mean()), vals, m.shape


res = pares["mask"].map(stats_mascara)
pares = pares.assign(area_tumor=[r[0] for r in res], valores=[r[1] for r in res], forma=[r[2] for r in res])
no_bin = pares.valores.map(lambda v: any(x not in (0, 255) for x in v))
print("Máscaras con valores intermedios (no estrictamente 0/255):", int(no_bin.sum()), "de", len(pares))
print("Ejemplos de valores:", Counter(pares.valores).most_common(3))
print(f"-> Se binarizan con umbral > {MASK_THR} (fondo = 0, tumor = 1) en todo el proyecto.")
print("\nTamaños de máscara:", Counter(pares.forma).most_common(3))
print("\nPorcentaje del tumor por clase (fracción de píxeles):")
display(pares.groupby("cls").area_tumor.describe().round(4))
print("Máscaras vacías (sin ningún píxel de tumor):", int((pares.area_tumor == 0).sum()))

## 4. Dimensiones de las imágenes

In [ ]:
dim = df_cls.groupby(["width", "height"]).size().sort_values(ascending=False)
print("Combinaciones de (ancho, alto):", len(dim))
display(dim.head(10))
print("Relación de aspecto única:", df_cls.eval("width / height").round(2).nunique() == 1)

## 5. Gráficos

In [ ]:
fig_dir = OUT / "figuras_eda"
fig_dir.mkdir(exist_ok=True)
colors = {"train": "#2F5597", "test": "#ED7D31"}

# (a) imágenes por clase y partición
fig, ax = plt.subplots(figsize=(7, 4))
tab = df_cls.groupby(["cls", "split"]).size().unstack(fill_value=0).reindex(CLASS_ORDER)
tab.rename(index=CLASS_ES).plot.bar(ax=ax, color=[colors.get(c, "gray") for c in tab.columns])
ax.set_title("Imágenes por clase y partición"); ax.set_xlabel(""); ax.set_ylabel("imágenes")
plt.xticks(rotation=20); plt.tight_layout(); fig.savefig(fig_dir / "conteo_clase_particion.png", dpi=130); plt.show()

# (b) planos de corte por clase
fig, ax = plt.subplots(figsize=(7, 4))
tab = pd.crosstab(df_cls.cls, df_cls.plane).reindex(CLASS_ORDER)
tab.rename(index=CLASS_ES, columns=PLANE_ES).plot.bar(ax=ax)
ax.set_title("Planos de corte por clase"); ax.set_xlabel(""); ax.set_ylabel("imágenes")
plt.xticks(rotation=20); plt.tight_layout(); fig.savefig(fig_dir / "planos_por_clase.png", dpi=130); plt.show()

# (c) tamaño del tumor (fracción de la imagen) por clase
fig, ax = plt.subplots(figsize=(7, 4))
for c in ["glioma", "meningioma", "pituitary"]:
    s = pares[pares.cls == c].area_tumor
    if len(s):
        ax.hist(s, bins=30, alpha=0.55, label=CLASS_ES[c])
ax.set_title("Tamaño del tumor (fracción de la imagen)"); ax.set_xlabel("fracción de píxeles de tumor"); ax.legend()
plt.tight_layout(); fig.savefig(fig_dir / "tamano_tumor.png", dpi=130); plt.show()
print("Figuras guardadas en", fig_dir)

## 6. Galería: imagen y máscara por clase

In [ ]:
rng = np.random.default_rng(SEED)
tumor_classes = ["glioma", "meningioma", "pituitary"]
fig, axes = plt.subplots(4, 3, figsize=(9, 12))
for r, c in enumerate(CLASS_ORDER):
    pool = df_cls[(df_cls.cls == c) & (df_cls.split == "train")]
    pick = pool.sample(min(3, len(pool)), random_state=SEED)
    for k, (_, row) in enumerate(pick.iterrows()):
        ax = axes[r, k]
        img = np.asarray(Image.open(ROOT / row.path).convert("RGB"))
        ax.imshow(img)
        m = pares[pares.stem == row.stem]
        if len(m):
            mk = np.asarray(Image.open(ROOT / m.iloc[0]["mask"]).convert("L")) > MASK_THR
            ov = np.zeros((*mk.shape, 4)); ov[..., 0] = 1; ov[..., 3] = 0.45 * mk
            ax.imshow(ov)
        ax.set_title(f"{CLASS_ES[c]} · {PLANE_ES.get(row.plane, row.plane)}", fontsize=9); ax.axis("off")
plt.tight_layout(); fig.savefig(fig_dir / "galeria_imagen_mascara.png", dpi=120); plt.show()

## 7. manifest.csv y duplicados exactos

In [ ]:
man_paths = sorted(ROOT.rglob("manifest.csv"))
if man_paths:
    man = pd.read_csv(man_paths[0])
    print("Columnas del manifest:", list(man.columns))
    display(man.head(3))
    sha_col = next((c for c in man.columns if "sha" in c.lower()), None)
    split_col = next((c for c in man.columns if c.lower() == "split"), None)
    if sha_col:
        # solo imágenes de la tarea de clasificación (el manifest lista también la tarea de segmentación, que repite las mismas imágenes)
        sub = man
        if "task" in man.columns:
            sub = sub[sub["task"].astype(str).str.lower().str.contains("class")]
        if "is_mask" in man.columns:
            sub = sub[~sub["is_mask"].astype(str).str.lower().isin(["true", "1"])]
        print("Imágenes de clasificación en el manifest:", len(sub))
        dup = sub[sub.duplicated(sha_col, keep=False)]
        print("Archivos con SHA-256 repetido dentro de clasificación (duplicados exactos):", len(dup))
        if split_col and len(dup):
            cruzados = dup.groupby(sha_col)[split_col].nunique()
            print("Duplicados exactos repartidos entre entrenamiento y prueba:", int((cruzados > 1).sum()))
else:
    print("No se encontró manifest.csv")

## 8. Resumen de hallazgos (para la presentación y el informe)

In [ ]:
resumen = {
    "imagenes_clasificacion": int(len(df_cls)),
    "entrenamiento": int((df_cls.split == "train").sum()),
    "prueba": int((df_cls.split == "test").sum()),
    "por_clase": df_cls.cls.value_counts().to_dict(),
    "pares_segmentacion": int(len(pares)),
    "pares_segmentacion_articulo": EXPECTED_SEG_PAIRS,
    "sin_tumor_con_mascara": int(len(sin_tumor_cls & con_mascara)),
    "mascaras_vacias": int((pares.area_tumor == 0).sum()),
    "diferencias_vs_articulo_clase_particion": {f"{k[0]}|{k[1]}": int(v) for k, v in cmp_cls["diferencia"].items()},
    "dimensiones_mas_frecuentes": [(int(w), int(h), int(n)) for (w, h), n in dim.head(3).items()],
}
(OUT / "eda_resumen.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(resumen, ensure_ascii=False, indent=2))

print("\nLIMITACIONES A REGISTRAR")
print("- El dataset no trae ID de paciente: imágenes de un mismo paciente pueden estar en una misma partición.")
print("- Solo hay imágenes T1 con contraste: los modelos pueden rendir peor con otras secuencias.")
print("- Licencia CC BY 4.0 (metadatos de Kaggle): citar a los autores de BRISC y a las fuentes originales del dataset.")
print("- Las máscaras no son estrictamente binarias: se binarizan con umbral > 127.")
print("- Las imágenes no tienen un único tamaño: el cargador las rellena a cuadrado (sin deformar) antes de redimensionar.")